In [1]:
import os
import itertools
import warnings
import numpy as np
import pandas as pd
from scipy.stats import spearmanr
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from sklearn.metrics import roc_auc_score, average_precision_score
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
 
warnings.filterwarnings("ignore")
B = int(os.environ.get("B_BOOT", 100))
S = int(os.environ.get("SEEDS", 5))
KIND = os.environ.get("BIN_MODEL", "xgb")
FS = os.environ.get("BIN_FS", "full")
KS = [1, 2, 3, 4, 5]
os.makedirs("results", exist_ok=True)
 
# ------------------------------------------------------------------ data
df = pd.read_csv("data/processed/features.csv")
g1 = pd.read_csv("results/feature_groups.csv", index_col=0)["group"]
G2 = g1.copy()
for c in ("mv_tl", "mv_ta", "log_mv"):
    G2[c] = "market"
for c in ("log_assets", "log_sales"):
    G2[c] = "size"
G2.rename("group").rename_axis("feature").to_csv("results/step6_groups_v2.csv")
GO = ["liquidity", "leverage", "profitability", "activity", "market", "size",
      "dynamics"]
ALL = g1.index.tolist()
COLS = ALL if FS == "full" else [c for c in ALL if g1[c] != "dynamics"]
tr = df[df.split == "train"].reset_index(drop=True)
va = df[df.split == "val"].reset_index(drop=True)
E = df[df.split.isin(["val", "test"])].reset_index(drop=True)
codes, uniq = pd.factorize(E.company_name)
nU = len(uniq)
rng = np.random.default_rng(42)
W = rng.multinomial(nU, np.full(nU, 1 / nU), size=B).astype(float)
 
 
def gshare(v, cols):
    M = np.array([[G2[c] == g for g in GO] for c in cols], float)
    s = np.atleast_2d(v) @ M
    return s / s.sum(1, keepdims=True)
 
 
def gsize(cols):
    return np.array([sum(G2[c] == g for c in cols) for g in GO], float)
 
 
def rho(a, b):
    return float(spearmanr(a, b)[0])
 
 
# ------------------------------------------------- models / contributions
def fit_bin(kind, Xtr, ytr, Xva, yva, seed):
    if kind == "xgb":
        from xgboost import XGBClassifier
        spw = float((ytr == 0).sum() / max((ytr == 1).sum(), 1))
        m = XGBClassifier(n_estimators=800, learning_rate=0.03, max_depth=4,
                          subsample=0.8, colsample_bytree=0.8,
                          min_child_weight=5, reg_lambda=2.0,
                          scale_pos_weight=spw, eval_metric="aucpr",
                          early_stopping_rounds=50, tree_method="hist",
                          n_jobs=-1, random_state=seed)
        m.fit(Xtr, ytr, eval_set=[(Xva, yva)], verbose=False)
        return m
    # other kinds: seed = random 85% subsample of the training rows
    r = np.random.default_rng(seed)
    keep = r.random(len(ytr)) < 0.85
    Xs, ys = Xtr[keep], ytr[keep]
    if kind == "ebm":
        from interpret.glassbox import ExplainableBoostingClassifier
        w = np.where(ys == 1, (ys == 0).sum() / max((ys == 1).sum(), 1), 1.0)
        m = ExplainableBoostingClassifier(random_state=seed)
        m.fit(Xs, ys, sample_weight=w)
        return m
    m = make_pipeline(SimpleImputer(strategy="median"), StandardScaler(),
                      LogisticRegression(C=0.1, class_weight="balanced",
                                         max_iter=3000))
    m.fit(Xs, ys)
    return m
 
 
def contrib(kind, model, X):
    if kind == "logreg":
        imp, sc, lr = (s[1] for s in model.steps)
        return sc.transform(imp.transform(X)) * lr.coef_[0]
    if kind == "xgb":
        import xgboost as xgb
        try:
            ir = (0, int(model.best_iteration) + 1)
        except Exception:
            ir = (0, 0)
        c = model.get_booster().predict(xgb.DMatrix(X, missing=np.nan),
                                        pred_contribs=True,
                                        iteration_range=ir)
        return c[:, :-1]
    T = model.eval_terms(X)
    out = np.zeros((len(X), X.shape[1]))
    for j, feats in enumerate(model.term_features_):
        for f in feats:
            out[:, f] += T[:, j] / len(feats)
    return out
 
 
def masks(k, d_tr=None):
    d_tr = tr if d_tr is None else d_tr
    mt = ((d_tr.failed_co == 0) | (d_tr.yrs_before_fail == k)).values
    mv = ((va.failed_co == 0) | (va.yrs_before_fail == k)).values
    me = ((E.failed_co == 0) | (E.yrs_before_fail == k)).values
    return mt, mv, me
 
 
def fit_k(k, cols, seed, d_tr=None):
    d_tr = tr if d_tr is None else d_tr
    mt, mv, me = masks(k, d_tr)
    ytr = (d_tr.yrs_before_fail[mt] == k).astype(int).values
    yva = (va.yrs_before_fail[mv] == k).astype(int).values
    model = fit_bin(KIND, d_tr.loc[mt, cols], ytr, va.loc[mv, cols], yva, seed)
    return model, me
 
 
def pos_importance(model, cols, k):
    X = E.loc[(E.yrs_before_fail == k).values, cols]
    return np.abs(contrib(KIND, model, X)).mean(0)
 
 
# ====================================================== 6b seed noise floor
print(f"model={KIND}, features={FS}, seeds={S}")
imp_rows, grp_rows, floor_rows, avg_imp = [], [], [], {}
for k in KS:
    I = np.array([pos_importance(fit_k(k, COLS, s)[0], COLS, k)
                  for s in range(S)])
    sh = I / I.sum(1, keepdims=True)
    gs = gshare(I, COLS)
    avg_imp[k] = I.mean(0)
    for j, c in enumerate(COLS):
        imp_rows.append({"k": k, "feature": c, "group": G2[c],
                         "share_mean": sh[:, j].mean(),
                         "share_sd": sh[:, j].std(ddof=1) if S > 1 else np.nan})
    n = gsize(COLS)
    for j, g in enumerate(GO):
        grp_rows.append({"k": k, "group": g, "n_features": int(n[j]),
                         "share_mean": gs[:, j].mean(),
                         "share_sd": gs[:, j].std(ddof=1) if S > 1 else np.nan,
                         "share_per_feature": gs[:, j].mean() / max(n[j], 1)})
    pr = [rho(I[i], I[j]) for i, j in itertools.combinations(range(S), 2)]
    floor_rows.append({"k": k, "seed_spearman_mean": np.mean(pr) if pr else np.nan,
                       "seed_spearman_min": np.min(pr) if pr else np.nan})
    print(f"  k={k} seeds done")
pd.DataFrame(imp_rows).to_csv("results/step6_seed_importance.csv", index=False)
gdf = pd.DataFrame(grp_rows)
gdf.to_csv("results/step6_seed_groups.csv", index=False)
floor = pd.DataFrame(floor_rows)
floor.to_csv("results/step6_seed_noise_floor.csv", index=False)
 
rc = [{"k_a": a, "k_b": b, "spearman_seedavg": rho(avg_imp[a], avg_imp[b])}
      for a, b in itertools.combinations(KS, 2)]
rc = pd.DataFrame(rc)
rc.to_csv("results/step6_bins_rank_corr_seedavg.csv", index=False)
 
fig, ax = plt.subplots(figsize=(6.5, 4))
for g in GO:
    d = gdf[gdf.group == g]
    ax.errorbar(d.k, d.share_mean, yerr=d.share_sd, marker="o", capsize=3,
                label=g)
ax.set_xlabel("years before bankruptcy filing")
ax.set_ylabel("share of mean |SHAP| (mean ± SD over seeds)")
ax.legend(fontsize=7, ncol=2)
fig.tight_layout()
fig.savefig("results/fig_group_share_seedavg.png", dpi=200)
 
# ====================================================== 6c period retraining
rows = []
P1, P2 = tr[tr.year <= 2005].reset_index(drop=True), \
    tr[(tr.year >= 2006)].reset_index(drop=True)
for k in KS:
    vs = []
    for P in (P1, P2):
        if ((P.yrs_before_fail == k).sum() < 10):
            vs.append(None)
            continue
        m, _ = fit_k(k, COLS, 0, P)
        vs.append(pos_importance(m, COLS, k))
    if vs[0] is None or vs[1] is None:
        continue
    rows.append({"k": k, "pos_train_1999_2005": int((P1.yrs_before_fail == k).sum()),
                 "pos_train_2006_2011": int((P2.yrs_before_fail == k).sum()),
                 "spearman_P1_vs_P2": rho(vs[0], vs[1])})
pd.DataFrame(rows).to_csv("results/step6_period_retrain.csv", index=False)
period = pd.DataFrame(rows)
 
# ====================================================== 6d group ablation
def wmetrics(y, s, w):
    return (roc_auc_score(y, s, sample_weight=w),
            average_precision_score(y, s, sample_weight=w))
 
 
rows = []
for k in KS:
    base_model, me = fit_k(k, COLS, 0)
    ye = (E.yrs_before_fail[me] == k).astype(int).values
    cu = codes[me]
    sb = base_model.predict_proba(E.loc[me, COLS])[:, 1]
    variants = {"(none)": sb}
    for g in GO:
        keep = [c for c in COLS if G2[c] != g]
        if len(keep) == len(COLS):
            continue
        m, _ = fit_k(k, keep, 0)
        variants[g] = m.predict_proba(E.loc[me, keep])[:, 1]
    ones = np.ones(len(ye))
    base_roc, base_pr = wmetrics(ye, sb, ones)
    bw = [W[b][cu] for b in range(B)]
    base_b = [wmetrics(ye, sb, w) for w in bw if (w[ye == 1] > 0).any()]
    for name, s in variants.items():
        roc, pr = wmetrics(ye, s, ones)
        r = {"k": k, "dropped_group": name, "ROC_AUC": roc, "PR_AUC": pr}
        if name != "(none)":
            bs = [wmetrics(ye, s, w) for w in bw if (w[ye == 1] > 0).any()]
            n = min(len(bs), len(base_b))
            dR = np.array([bs[i][0] - base_b[i][0] for i in range(n)])
            dP = np.array([bs[i][1] - base_b[i][1] for i in range(n)])
            r["dROC"] = roc - base_roc
            r["dROC_lo"], r["dROC_hi"] = np.percentile(dR, [2.5, 97.5])
            r["dPR"] = pr - base_pr
            r["dPR_lo"], r["dPR_hi"] = np.percentile(dP, [2.5, 97.5])
        rows.append(r)
    print(f"  ablation k={k} done")
abl = pd.DataFrame(rows)
abl.to_csv("results/step6_ablation.csv", index=False)
 
# ====================================== 6e paired comparison between models
pred = pd.read_csv("data/processed/preds.csv")
pred = pred[(pred.split == "test") & (pred.features.isin(["full"]))]
te_codes = None
rows = []
for h in (1, 2, 3):
    ph = pred[pred.horizon == h]
    models = sorted(ph.model.unique())
    if len(models) < 2:
        continue
    ref = ph[ph.model == models[0]].reset_index(drop=True)
    cc, _ = pd.factorize(ref.company_name)
    nC = cc.max() + 1
    r2 = np.random.default_rng(7)
    Wt = r2.multinomial(nC, np.full(nC, 1 / nC), size=B).astype(float)
    y = ref.y.values
    sc = {m: ph[ph.model == m].reset_index(drop=True).score.values
          for m in models}
    for a, b in itertools.combinations(models, 2):
        for name, fn in (("PR_AUC", average_precision_score),
                         ("ROC_AUC", roc_auc_score)):
            d = []
            for i in range(B):
                w = Wt[i][cc]
                if not (w[y == 1] > 0).any():
                    continue
                d.append(fn(y, sc[a], sample_weight=w) -
                         fn(y, sc[b], sample_weight=w))
            d = np.array(d)
            rows.append({"horizon": h, "model_a": a, "model_b": b,
                         "metric": name,
                         "delta_a_minus_b": fn(y, sc[a]) - fn(y, sc[b]),
                         "lo": np.percentile(d, 2.5),
                         "hi": np.percentile(d, 97.5)})
pairs = pd.DataFrame(rows)
pairs["excludes_zero"] = (pairs.lo > 0) | (pairs.hi < 0)
pairs.to_csv("results/step6_pairwise_models.csv", index=False)
 
# ------------------------------------------------------------------ summary
pd.set_option("display.width", 200)
print("\n=== Seed noise floor vs between-bin correlation ===")
print(floor.round(3).to_string(index=False))
print(rc.round(3).to_string(index=False))
print("\n=== Group share by lead time (seed mean, SD, per-feature) ===")
print(gdf.pivot(index="group", columns="k", values="share_mean")
      .reindex(GO).round(3).to_string())
print(gdf.pivot(index="group", columns="k", values="share_sd")
      .reindex(GO).round(3).to_string())
print(gdf.pivot(index="group", columns="k", values="share_per_feature")
      .reindex(GO).round(4).to_string())
print("\n=== Top 8 features by seed-mean share ===")
t = pd.DataFrame(imp_rows).pivot(index="feature", columns="k",
                                 values="share_mean")
print(t.assign(mx=t.max(axis=1)).sort_values("mx", ascending=False)
      .head(8).round(3).to_string())
print("\n=== Period retraining ===")
print(period.round(3).to_string(index=False))
print("\n=== Ablation (drop in ROC/PR vs full) ===")
print(abl[abl.dropped_group != "(none)"][
    ["k", "dropped_group", "dROC", "dROC_lo", "dROC_hi", "dPR", "dPR_lo",
     "dPR_hi"]].round(3).to_string(index=False))
print("\n=== Paired model comparison (test) ===")
print(pairs.round(3).to_string(index=False))
 

model=xgb, features=full, seeds=5
  k=1 seeds done
  k=2 seeds done
  k=3 seeds done
  k=4 seeds done
  k=5 seeds done
  ablation k=1 done
  ablation k=2 done
  ablation k=3 done
  ablation k=4 done
  ablation k=5 done

=== Seed noise floor vs between-bin correlation ===
 k  seed_spearman_mean  seed_spearman_min
 1               0.918              0.887
 2               0.850              0.750
 3               0.863              0.790
 4               0.745              0.480
 5               0.783              0.697
 k_a  k_b  spearman_seedavg
   1    2             0.530
   1    3             0.393
   1    4             0.369
   1    5             0.102
   2    3             0.698
   2    4             0.563
   2    5             0.394
   3    4             0.637
   3    5             0.547
   4    5             0.579

=== Group share by lead time (seed mean, SD, per-feature) ===
k                  1      2      3      4      5
group                                           
liquidi